# OER Workflow via Explog + JobHandler to OH Values

This notebook demonstrates an end-to-end workflow using `Explog` and `Job_handler` in `mode="test"`.
It runs bulk, surface, O-adsorption, and OH-adsorption via process submission
to obtain OH-related values.

In [11]:
from pathlib import Path
import sys
import shutil

import pandas as pd
from ase.io import read

In [12]:
# Ensure local package import works when running from notebooks/
repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from gnome_dreams_oer_screening import Explog, OER_catalyst_study

In [13]:
# Load test bulk structure
bulk_atoms = read(repo_root / "notebooks" / "TiO2_rutile.cif")
bulk_atoms

Atoms(symbols='Ti2O4', pbc=True, cell=[4.59983732, 4.59983732, 2.95921356], spacegroup_kinds=...)

In [14]:
# Initialize EXPLOG and add one candidate
work_dir = repo_root / "notebooks" / "TEMP_explog_oer_workflow"
if work_dir.exists():
    shutil.rmtree(work_dir)
work_dir.mkdir(parents=True, exist_ok=True)

EXPLOG = Explog()
EXPLOG.init(work_dir, mode="test", prefix_vasp_dirs=False)

candidate_id = "demo_candidate_1"
study = OER_catalyst_study(
    bulk_atoms.copy(),
    H2O_gas_free_energy=-14.217,
    H2_gas_free_energy=-6.77,
)
EXPLOG.add_candidate(candidate_id=candidate_id, study_obj=study)

EXPLOG.relational_frame.candidates.df

,candidate_id,reason_or_hypothesis,notes,study_obj,Overpotential_from_scaling,idealOverPotential
0,demo_candidate_1,None,None,<gnome_dreams_oer_screening.oer.oer_study.OER_...,<NA>,<NA>


In [15]:
# Run bulk relaxation via process API
p_bulk = EXPLOG.add_process(candidate_id=candidate_id, calculation_type="bulk_relaxation")
EXPLOG.submit_process(p_bulk)
updates = EXPLOG.update_log()
print("updates:", updates)

cand_study = EXPLOG.relational_frame.candidates[candidate_id].study_obj
print("bulk set:", cand_study.get_bulk_atoms() is not None)

updates: {'0': 'completed'}
bulk set: True


In [16]:
# Rank surfaces, initialize one termination, and run surface relaxation via process API
cand_study.predict_most_likely_surfaces(max_miller=1, method="O-coordination")
ranking = cand_study.get_termination_rankings().sort_values("Normalized score", ascending=False)
termination_index = int(ranking.iloc[0]["Termination index"])
cand_study.initialize_oer_surface_study(termination_index)

p_surface = EXPLOG.add_process(
    candidate_id=candidate_id,
    calculation_type="surface_relaxation",
    termination_index=termination_index,
)
EXPLOG.submit_process(p_surface)
updates = EXPLOG.update_log()
print("updates:", updates)
ranking[["Termination index", "Miller indices", "Normalized score"]].head()

updates: {'1': 'completed'}


,Termination index,Miller indices,Normalized score
2,2,"(1, 1, 0)",1.000000
4,4,"(1, 0, 0)",0.593465
3,3,"(1, 0, 1)",0.479541
5,5,"(0, 0, 1)",0.195581
1,1,"(1, 1, 1)",0.000000


In [17]:
# Determine adsorption sites and initialize one adsorption-site study
surface_study = cand_study.get_surface_studies()[termination_index]
surface_study.determine_adsorption_sites(only_reduced_coord_O_sites=True)
sites_df = surface_study.get_adsorption_sites_df()

if (sites_df["site type"] == "on-top").any():
    site_index = int(sites_df[sites_df["site type"] == "on-top"]["Site index"].iloc[0])
else:
    site_index = int(sites_df["Site index"].iloc[0])

surface_study.initialize_adsorption_site_study(site_index)
sites_df[["Site index", "site type", "ad site element"]]

,Site index,site type,ad site element
0,0,on-top,Ti
1,1,on-top,Ti
2,2,lattice O,N/A


In [18]:
# Run O adsorption via process API
p_o = EXPLOG.add_process(
    candidate_id=candidate_id,
    calculation_type="O_adsorption",
    termination_index=termination_index,
    site_index=site_index,
)
EXPLOG.submit_process(p_o)
updates = EXPLOG.update_log()
print("updates:", updates)
EXPLOG.relational_frame.processes.df.tail(3)

updates: {'2': 'completed'}


,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,demo_candidate_1,bulk_relaxation,42,completed,None,None,c:\Users\matnis\OneDrive - Danmarks Tekniske U...,None
1,1,demo_candidate_1,surface_relaxation,43,completed,2,None,c:\Users\matnis\OneDrive - Danmarks Tekniske U...,None
2,2,demo_candidate_1,O_adsorption,44,completed,2,0,c:\Users\matnis\OneDrive - Danmarks Tekniske U...,None


In [19]:
EXPLOG.relational_frame.candidates.df

,candidate_id,reason_or_hypothesis,notes,study_obj,Overpotential_from_scaling,idealOverPotential
0,demo_candidate_1,None,None,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,0.84


In [20]:
# Run OH adsorption via process API
p_oh_list = EXPLOG.add_process(
    candidate_id=candidate_id,
    calculation_type="OH_adsorption",
    termination_index=termination_index,
    site_index=site_index,
)
for p_oh in p_oh_list:
    EXPLOG.submit_process(p_oh)
updates = EXPLOG.update_log()
print("updates:", updates)
EXPLOG.relational_frame.processes.df.tail(6)

updates: {'3': 'completed', '4': 'completed', '5': 'completed'}


,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,demo_candidate_1,bulk_relaxation,42,completed,None,None,c:\Users\matnis\OneDrive - Danmarks Tekniske U...,None
1,1,demo_candidate_1,surface_relaxation,43,completed,2,None,c:\Users\matnis\OneDrive - Danmarks Tekniske U...,None
2,2,demo_candidate_1,O_adsorption,44,completed,2,0,c:\Users\matnis\OneDrive - Danmarks Tekniske U...,None
3,3,demo_candidate_1,OH_adsorption,45,completed,2,0,c:\Users\matnis\OneDrive - Danmarks Tekniske U...,OH_init_index=0
4,4,demo_candidate_1,OH_adsorption,46,completed,2,0,c:\Users\matnis\OneDrive - Danmarks Tekniske U...,OH_init_index=1
5,5,demo_candidate_1,OH_adsorption,47,completed,2,0,c:\Users\matnis\OneDrive - Danmarks Tekniske U...,OH_init_index=2


In [21]:
# Update derived energies/overpotentials and inspect outputs
surface_study.update_adsorption_energies()
EXPLOG.job_handler._update_ideal_over_pot(candidate_id)

final_df = surface_study.get_adsorption_sites_df()
cols = [
    c for c in [
        "Site index",
        "site type",
        "G(O)",
        "G(OH)",
        "ideal overpotential",
        "overpotential from OH-OOH scaling relation",
    ]
    if c in final_df.columns
]
display(final_df.loc[[site_index], cols])

candidate_row = EXPLOG.relational_frame.candidates[candidate_id]
print("idealOverPotential:", candidate_row.idealOverPotential)
print("Overpotential_from_scaling:", candidate_row.Overpotential_from_scaling)
EXPLOG.relational_frame.candidates.df

,Site index,site type,G(O),G(OH),ideal overpotential,overpotential from OH-OOH scaling relation
0,0,on-top,0.789787,-22.893647,22.45,23.38


idealOverPotential: 22.45
Overpotential_from_scaling: 23.38


,candidate_id,reason_or_hypothesis,notes,study_obj,Overpotential_from_scaling,idealOverPotential
0,demo_candidate_1,None,None,<gnome_dreams_oer_screening.oer.oer_study.OER_...,23.38,22.45
